# Responder las preguntas en Colab (llama.cpp CUDA + Qwen3-8B)

Corre el pipeline completo (`src/main.py`) en una GPU de Colab: recuperacion hibrida (BM25 + bge-m3) + decoder Qwen3-8B Q4_K_M servido por `llama-server`.
No cambia ningun codigo del repo: solo prepara el entorno y lanza `python src/main.py`.

**Antes de empezar (en tu maquina, una sola vez)**
1. Sube la carpeta `data_corpus/indice/` (con `chunks.jsonl`, `index.faiss`, `indice_info.json` y `bm25/`, ~280 MB) a Drive en `Mi unidad/syntax/indice/`.
2. Para el sabado: sube `test_992.jsonl` a `Mi unidad/syntax/test_992.jsonl`.
3. La rama que vayas a usar debe estar **pusheada** a GitHub (el notebook hace `git clone`).

En Colab: Entorno de ejecucion -> Cambiar tipo -> **GPU** (T4 gratis; L4/A100 con Colab Pro).

**Orden:** 1 a 7 una vez por sesion; 8 = humo; 9 = sample + evaluacion; 10 = las 992. Las salidas y trazas se escriben en Drive (`syntax/salidas/`), asi una desconexion no las pierde y `main.py` reanuda donde quedo.

**Reproducibilidad:** la corrida final de las 992 y la verificacion en vivo deben usar el mismo GGUF (sha256 fijado en `src/config.py`) y la misma build de llama.cpp. La celda 7 imprime la build; anotala.

## 1. GPU, Drive y parametros

In [ ]:
import subprocess, os
from pathlib import Path
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], capture_output=True, text=True).stdout)
assert subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0, "Activar GPU: Entorno de ejecucion -> Cambiar tipo"

from google.colab import drive
drive.mount("/content/drive")

# ---- Parametros (editar aqui) ----
REPO_URL = "https://github.com/syntax-hackathon-2026/rag-juridico-syntax.git"
RAMA = "santiago"                      # rama pusheada con el pipeline de generacion
DRIVE = Path("/content/drive/MyDrive/syntax")
INDICE_DRIVE = DRIVE / "indice"        # copia de data_corpus/indice/
TEST_DRIVE = DRIVE / "test_992.jsonl"  # solo para el sabado
SALIDAS_DRIVE = DRIVE / "salidas"      # entregas y trazas (sobreviven a la sesion)
BIN_DRIVE = DRIVE / "bin"              # llama-server compilado (cache por arquitectura de GPU)
ROOT = Path("/content/repo")
for d in (SALIDAS_DRIVE, BIN_DRIVE):
    d.mkdir(parents=True, exist_ok=True)
assert (INDICE_DRIVE / "chunks.jsonl").exists(), f"falta {INDICE_DRIVE}/chunks.jsonl: subir data_corpus/indice/ a Drive"

## 2. Codigo del repo
Si el repo es privado, pega un token de GitHub (solo lectura) cuando lo pida; no se guarda.

In [ ]:
import getpass, shutil
if ROOT.exists():
    shutil.rmtree(ROOT)
token = getpass.getpass("Token de GitHub (Enter si el repo es publico): ").strip()
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
r = subprocess.run(["git", "clone", "-q", "-b", RAMA, url, str(ROOT)], capture_output=True, text=True)
print(r.stderr.replace(token, "***") if token else r.stderr)
r.check_returncode()
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=ROOT, check=True)   # no dejar el token en .git/config
print(subprocess.run(["git", "log", "--oneline", "-3"], cwd=ROOT, capture_output=True, text=True).stdout)
os.chdir(ROOT)

## 3. Dependencias de Python
Solo las del runtime (retrieval + generacion), con la version fijada en `requirements.txt`. `torch` y `numpy` ya vienen en Colab y no se reinstalan (el pin de numpy choca con `numba` de Colab; no afecta al pipeline). Si ves un aviso de pip por otro conflicto de dependencias, es solo un aviso. Colab trae Python 3.12 y el repo declara 3.13: no correr `verificar_deps.py` ni `preparar_entorno.py` aqui.

In [ ]:
import re
reqs = (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines()
runtime = ("bm25s", "faiss-cpu", "huggingface_hub", "jsonschema", "sentence-transformers")
pines = [l.strip() for l in reqs if re.match(rf"^({'|'.join(runtime)})==", l.strip())]
print(pines)
r = subprocess.run(["pip", "-q", "install", *pines], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])

## 4. Indice desde Drive
Se copia a disco local (leer `index.faiss` y `bm25/` a traves de Drive es lento) y se comprueba que es el congelado.

In [ ]:
import json, hashlib
destino = ROOT / "data_corpus" / "indice"
if destino.exists():
    shutil.rmtree(destino)
shutil.copytree(INDICE_DRIVE, destino)
info = json.loads((destino / "indice_info.json").read_text(encoding="utf-8"))
sha = hashlib.sha256((destino / "chunks.jsonl").read_bytes()).hexdigest()
esperado = json.loads((ROOT / "data" / "hashes_esperados.json").read_text(encoding="utf-8"))["sha256_chunks"]
assert sha == info["sha256_chunks"] == esperado, f"chunks.jsonl no es el congelado: {sha}"
bm25 = ["data.csc.index.npy", "indices.csc.index.npy", "indptr.csc.index.npy", "params.index.json", "vocab.index.json"]
faltan = [f for f in ["index.faiss", *[f"bm25/{x}" for x in bm25]] if not (destino / f).is_file()]
assert not faltan, f"faltan en Drive ({INDICE_DRIVE}): {faltan}. Subir data_corpus/indice/ completa"
print("indice ok:", info["n_fragmentos"], "fragmentos, sha256", sha[:12])

## 5. llama.cpp con CUDA
Se compila una vez por arquitectura de GPU (~5-10 min) como binario estatico y se guarda en Drive (`syntax/bin/`); las sesiones siguientes solo lo copian. Para usar exactamente la misma build que en otra maquina, fijar `LLAMA_REF` a un tag/commit.

In [ ]:
cc = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().splitlines()[0]
arch = cc.replace(".", "")                     # T4 -> 75, A100 -> 80, L4 -> 89
LLAMA_REF = None                               # p. ej. "b6000"; None = ultimo master (anotar la build que salga)
cache_bin = BIN_DRIVE / f"llama-server-sm{arch}"
if cache_bin.exists():
    shutil.copy(cache_bin, "/usr/local/bin/llama-server")
    os.chmod("/usr/local/bin/llama-server", 0o755)
    print("binario copiado desde Drive:", cache_bin.name)
else:
    src = Path("/content/llama.cpp")
    if src.exists():
        shutil.rmtree(src)
    clone = ["git", "clone", "-q", "https://github.com/ggml-org/llama.cpp", str(src)]
    subprocess.run(clone, check=True)
    if LLAMA_REF:
        subprocess.run(["git", "checkout", "-q", LLAMA_REF], cwd=src, check=True)
    subprocess.run(["cmake", "-B", "build", "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={arch}",
                    "-DBUILD_SHARED_LIBS=OFF", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"], cwd=src, check=True)
    subprocess.run(["cmake", "--build", "build", "-j", str(os.cpu_count()), "--target", "llama-server"], cwd=src, check=True)
    shutil.copy(src / "build" / "bin" / "llama-server", "/usr/local/bin/llama-server")
    shutil.copy(src / "build" / "bin" / "llama-server", cache_bin)
    print("compilado y guardado en", cache_bin)

## 6. Modelo (GGUF)
Baja el GGUF de Hugging Face con la revision fijada y verifica el sha256 (~5 GB, un par de minutos).

In [ ]:
r = subprocess.run(["python", "src/generacion/modelo.py", "descargar"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
r.check_returncode()

## 7. Levantar el servidor
Usa `modelo.py servir`, o sea los mismos flags que en local (`-np 1`, puerto 8081 porque Colab ya usa el 8080, `--load-mode none`, `--jinja`, `--temp 0`, `--seed 0`, ctx 8192, `-ngl 99`). El log queda en `/content/llama-server.log`.

In [ ]:
import time, urllib.request, sys
subprocess.run(["pkill", "-f", "llama-server"])          # por si quedo una instancia de una corrida anterior
time.sleep(2)
os.environ["SYNTAX_LLM_URL"] = "http://127.0.0.1:8081/v1"   # Colab usa el 8080

log = open("/content/llama-server.log", "w")
servidor = subprocess.Popen([sys.executable, "src/generacion/modelo.py", "servir"], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8081/health", timeout=2).status == 200:
            break
    except Exception:
        if servidor.poll() is not None:
            break
        time.sleep(2)
else:
    print("el servidor no respondio a tiempo")
print(open("/content/llama-server.log").read()[-1500:])
print(subprocess.run([sys.executable, "src/generacion/modelo.py", "verificar"], cwd=ROOT, capture_output=True, text=True).stdout)

## 8. Humo (3 preguntas)
Comprueba retriever + decoder de punta a punta y da la latencia por pregunta en esta GPU. Si pasa de ~20 s/pregunta, esta maquina no alcanza para las 992 en ~6 h.

In [ ]:
# salidas/ y trazas van a Drive para sobrevivir a una desconexion
salidas = ROOT / "salidas"
if salidas.exists() or salidas.is_symlink():
    shutil.rmtree(salidas) if salidas.is_dir() and not salidas.is_symlink() else salidas.unlink()
salidas.symlink_to(SALIDAS_DRIVE, target_is_directory=True)

# JAX (preinstalado en Colab, lo importa bm25s) intenta reservar ~75 % de la GPU y compite con llama-server
env = {**os.environ, "SYNTAX_DEVICE": "cuda", "PYTHONUNBUFFERED": "1",
       "JAX_PLATFORMS": "cpu", "XLA_PYTHON_CLIENT_PREALLOCATE": "false"}
def correr(*args):
    p = subprocess.Popen([sys.executable, *args], cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for linea in p.stdout:
        print(linea, end="")
    return p.wait()

correr("src/main.py", "--split", "sample", "--experimento", "colab_humo", "--limite", "3", "--sin-reanudar")
import statistics
filas = [json.loads(l) for l in open(salidas / "sample_colab_humo.jsonl", encoding="utf-8")]
lat = [f["latencia_ms"] / 1000 for f in filas if f.get("latencia_ms")]
print(f"latencia por pregunta: {lat}  media={statistics.mean(lat):.1f}s")

## 9. Sample (50 preguntas) + evaluacion oficial
Anota la fila de resultados en `evaluation/experiments.csv` copiando `evaluation/generacion/<exp>/` al repo (celda siguiente). Cambiar `EXP` para cada corrida.

In [ ]:
EXP = "colab_t4_v0"
correr("src/main.py", "--split", "sample", "--experimento", EXP)
correr("src/evaluacion/evaluar_entrega.py", "--entrega", f"salidas/sample_{EXP}.jsonl", "--experimento", EXP)

In [ ]:
# Respaldo de lo que el evaluador deja en el repo clonado (trazas, reporte, errores.csv, experiments.csv) -> Drive
dest = SALIDAS_DRIVE / "evaluation" / EXP
dest.mkdir(parents=True, exist_ok=True)
for p in (ROOT / "evaluation" / "generacion" / EXP).glob("*"):
    shutil.copy(p, dest / p.name)
shutil.copy(ROOT / "evaluation" / "experiments.csv", SALIDAS_DRIVE / "evaluation" / "experiments.csv")
print("copiado a", dest)

## 10. Las 992 (sabado)
Reanuda sola: si la sesion se cae, vuelve a correr 1-7 y esta celda; salta los ids ya escritos. La entrega queda en Drive: `syntax/salidas/submissions.jsonl`.

**Varias sesiones en paralelo:** cada persona abre una copia de este notebook con su `PARTICION` (`"1/3"`, `"2/3"`, `"3/3"`); cada una escribe `submissions_<I>_de_<N>.jsonl`. Despues hay que concatenarlas **en el orden de entrada** y validar contra el schema (celda siguiente). Todas deben usar la misma GPU/build/GGUF.

In [ ]:
PARTICION = None          # None = todo en esta sesion; o "1/3"
shutil.copy(TEST_DRIVE, ROOT / "data" / "test_992.jsonl")
sufijo = "" if not PARTICION else "_" + PARTICION.replace("/", "_de_")
salida = SALIDAS_DRIVE / f"submissions{sufijo}.jsonl"
args = ["src/main.py", "--split", "test", "--experimento", f"final{sufijo}", "--salida", str(salida)]
if PARTICION:
    args += ["--particion", PARTICION]
correr(*args)

In [ ]:
# Si se corrio en particiones: unir en el orden de data/test_992.jsonl y validar contra el schema
import glob
sys.path.insert(0, str(ROOT / "scripts"))
from common import read_jsonl
partes = {}
for f in sorted(glob.glob(str(SALIDAS_DRIVE / "submissions_*_de_*.jsonl"))):
    for fila in read_jsonl(Path(f)):
        partes[fila["id"]] = fila
if partes:
    orden = [it["id"] for it in read_jsonl(ROOT / "data" / "test_992.jsonl")]
    faltan = [i for i in orden if i not in partes]
    print(len(partes), "respuestas; faltan", len(faltan))
    if not faltan:
        with open(SALIDAS_DRIVE / "submissions.jsonl", "w", encoding="utf-8", newline="\n") as fh:
            for i in orden:
                fh.write(json.dumps(partes[i], ensure_ascii=False) + "\n")
        print("escrito", SALIDAS_DRIVE / "submissions.jsonl")

## Notas
- `torch`, `numpy` y `sentence-transformers` de Colab pueden diferir de los pines de `requirements.txt`; los embeddings del corpus ya estan calculados (solo se codifica la consulta, siempre fp32), pero **anota la version de torch/CUDA** (`registrar_entorno.py --salida ...`) en `experiments.csv`.
- El GGUF Q4_K_M en CUDA no da necesariamente la misma salida que en Mac/Metal: la corrida final y la verificacion en vivo deben ser en la misma maquina y build.
- Si hay OOM al cargar el modelo, reiniciar el entorno de ejecucion (la T4 no debe tener nada mas cargado).